# Scrapping the Indian Econometric Society
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Installing and loading all dependencies and libraries needed

In [2]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.7/481.7 kB 27.8 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,192 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,525 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics

# Scrapping ISI: 3rd attempt: PDF MINER.SIX

## Extracting text

In [3]:
!pip install pdfminer.six
from pdfminer.high_level import extract_text

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 44.1 MB/s eta 0:00:00


In [4]:
# Specify the path to your PDF in Google Drive
pdf_path = '/content/drive/My Drive/ISLE_2024.pdf'

In [ ]:
text = extract_text(pdf_path)
print(text)


List of the Life Members of TIES 
(The addresses with red background indicate changed/incorrect addresses from where the 
communications are returned back. Any change in the address or if the name is missing, it 
may be informed to the Treasurer, TIES, at tiestreasurer@gmail.com) 

S. N.  Name of the Life Members 

S. N.  Name of the Life Members 

1. 

3. 

5. 

7. 

9. 

L.2169 (96-97) 
Badr Abdi 
83/III Jai Jawan Colony 
JLN Marg Jaipur 302 018 

L.1120 (89-90) 
Shabd.S. Acharya 
Honorary Professor 
33, Shashi Complex, Sector-11 
Udaipur 
Udaipur 313 002, Rajasthan 
L.2080 (91-92) 
Narinderjit Singh Ahluwalia 
Lyallpur Khalsa College Campus 
G. T. Road, Jalandhar 144 001 

L.2170 (91-92) 
Rajesh Kumar Verma 
93-Nalapar, Purbia Tola  
Etawah, UP 

L.313 (74-75) 
F.A. Saiyed Bagh E Ahmedi 
9, Roshan Park Society 
Jivaraj Park, Ahmedabad 380 051 

11. 

L.829 (85-86) 
Ashok Parikh 
Department of Economics 
University of East Anglia 
Norwich NR4 7TI 
United Kingdom 

2. 

4. 

6. 

8. 


In [5]:
import re
import pandas as pd
from pdfminer.high_level import extract_text
import spacy

# Load spaCy NLP model
nlp = spacy.load("en_core_web_sm")

def extract_pdf_text(pdf_path):
    """Extract text from a PDF file."""
    return extract_text(pdf_path)

def parse_sessions(text):
    """Parse technical sessions into a structured DataFrame."""
    session_data = []
    session_blocks = re.split(r"Day \d+ .*", text)

    for block in session_blocks:
        if "Technical Session" in block:
            lines = block.split("\n")
            current_session = {}
            for line in lines:
                if "Technical Session" in line:
                    current_session['Session'] = line.strip()
                elif "Venue:" in line:
                    current_session['Venue'] = line.replace("Venue:", "").strip()
                elif "Chair:" in line:
                    current_session['Chair'] = line.replace("Chair:", "").strip()
                elif re.search(r"\bDiscussant\d?:", line):
                    if 'Discussants' not in current_session:
                        current_session['Discussants'] = []
                    current_session['Discussants'].append(line.replace("Discussant:", "").strip())
                elif ":" in line and not current_session.get("Title of Paper"):
                    current_session['Title of Paper'] = line.strip()
                elif line.strip():
                    current_session['Details'] = current_session.get('Details', '') + " " + line.strip()
            if current_session:
                session_data.append(current_session)

    return pd.DataFrame(session_data)

def parse_plenaries(text):
    """Parse plenary sessions into a structured DataFrame."""
    plenary_data = []
    plenary_blocks = re.split(r"Plenary Session \d+:?", text)

    for block in plenary_blocks:
        lines = block.split("\n")
        current_plenary = {}
        for line in lines:
            if "Chair:" in line:
                current_plenary['Chair'] = line.replace("Chair:", "").strip()
            elif "Venue:" in line:
                current_plenary['Venue'] = line.replace("Venue:", "").strip()
            elif "Title:" in line:
                current_plenary['Title'] = line.replace("Title:", "").strip()
            elif "Lecture by:" in line:
                current_plenary['Speaker'] = line.replace("Lecture by:", "").strip()
        if current_plenary:
            plenary_data.append(current_plenary)

    return pd.DataFrame(plenary_data)

def parse_panels(text):
    """Parse parallel panels into a structured DataFrame."""
    panel_data = []
    panel_blocks = re.split(r"Panel \d+:?", text)

    for block in panel_blocks:
        lines = block.split("\n")
        current_panel = {}
        for line in lines:
            if "Chair:" in line:
                current_panel['Chair'] = line.replace("Chair:", "").strip()
            elif "Venue:" in line:
                current_panel['Venue'] = line.replace("Venue:", "").strip()
            elif ":" in line and not current_panel.get("Topic"):
                current_panel['Topic'] = line.strip()
            elif "Organiser:" in line:
                current_panel['Organiser'] = line.replace("Organiser:", "").strip()
        if current_panel:
            panel_data.append(current_panel)

    return pd.DataFrame(panel_data)

def parse_roundtables(text):
    """Parse roundtable discussions into a structured DataFrame."""
    roundtable_data = []
    roundtable_blocks = re.split(r"Roundtable:", text)

    for block in roundtable_blocks:
        lines = block.split("\n")
        current_roundtable = {}
        for line in lines:
            if "Venue:" in line:
                current_roundtable['Venue'] = line.replace("Venue:", "").strip()
            elif "Topic:" in line:
                current_roundtable['Topic'] = line.replace("Topic:", "").strip()
            elif "Organiser:" in line:
                current_roundtable['Organiser'] = line.replace("Organiser:", "").strip()
        if current_roundtable:
            roundtable_data.append(current_roundtable)

    return pd.DataFrame(roundtable_data)

# Load PDF and extract text
pdf_path = "/content/drive/My Drive/ISLE_2024.pdf"  # Replace with your file path
text = extract_pdf_text(pdf_path)

# Parse into DataFrames
df_one = parse_sessions(text)
df_two = parse_plenaries(text)
df_three = parse_panels(text)
df_four = parse_roundtables(text)

# Save or display the DataFrames
df_one.to_csv("sessions.csv", index=False)
df_two.to_csv("plenaries.csv", index=False)
df_three.to_csv("panels.csv", index=False)
df_four.to_csv("roundtables.csv", index=False)

print("Technical Sessions:", df_one.head(), sep="\n")
print("Plenary Sessions:", df_two.head(), sep="\n")
print("Parallel Panels:", df_three.head(), sep="\n")
print("Roundtables:", df_four.head(), sep="\n")


Technical Sessions:
                                      Title of Paper  \
0               PRE-CONFERENCE EVENTS: 28 MARCH 2024   
1  Food Security, Informal Employment and Precari...   
2  Caste in Gig Economy: Exploring the ‘Spatialit...   
3  Spatial Inequalities in Household Earnings in ...   
4  Transcending Boundaries: Global Value Chains, ...   

                                             Details  \
0   The Indian Society of Labour Economics 64th L...   
1   Infrastructure and Emergent Welfare Framework...   
2   Damni Kain, Lecturer, O.P. Jindal Global Univ...   
3   Rural-Urban Differences S Chandrasekhar, Prof...   
4   Outcomes Sugata Marjit, Professor, Indian Ins...   

                                             Session  \
0  Technical Session 3.10: Care Economy and Labou...   
1                              Technical Session 1.1   
2                              Technical Session 1.2   
3                              Technical Session 2.1   
4                         

In [ ]:
print(df)

                     Name              Affiliation  \
0             S. N.  Name                     None   
1             S. N.  Name                     None   
2               Badr Abdi                   L.2169   
3              S. Acharya       Rajasthan \nL.2080   
4      Rajesh Kumar Verma                     None   
..                    ...                      ...   
957  Yellukunte Extension        Bommanahalli Post   
958     Harikrishnan K.S.            IIT Kharagpur   
959         Apoorva Verma                     None   
960   Bhanu Pratap Pandey                     None   
961           Manu Sharma  Department of Economics   

                                               Address  \
0                                  of the Life Members   
1                                  of the Life Members   
2    L.2169 (96-97) 83/III Jai Jawan Colony JLN Mar...   
3    L.1120 (89-90) Shabd. Honorary Professor 33, S...   
4    L.2170 (91-92) 93-Nalapar, Purbia Tola Etawah, UP   
.. 

Clean the df

In [ ]:
def clean_and_update_affiliations(df):
    """
    Clean and update the DataFrame:
    1. Drop rows where 'Name' is 'S. N. Name'.
    2. Append university names from 'Address' to 'Affiliation' where applicable.
    """
    # Drop rows where 'Name' is 'S. N. Name'
    df = df[df['Name'] != 'S. N. Name']

    # NLP model for processing Address
    nlp = spacy.load("en_core_web_sm")

    # Function to find and append university names
    def append_university(row):
        address = row['Address']
        affiliation = row['Affiliation'] or ""

        if pd.isna(address):
            return affiliation  # If no address, return existing affiliation

        doc = nlp(address)
        universities = [ent.text for ent in doc.ents if "university" in ent.text.lower()]

        if universities:
            # Append found universities to the affiliation
            affiliation = f"{affiliation}, {', '.join(universities)}" if affiliation else ", ".join(universities)

        return affiliation

    # Update the 'Affiliation' column
    df['Affiliation'] = df.apply(append_university, axis=1)

    return df

# Clean and update the DataFrame
df_cleaned = clean_and_update_affiliations(df)

# Save or display the updated DataFrame
df_cleaned.to_csv("updated_members_list.csv", index=False)
print(df_cleaned.head())


                 Name         Affiliation  \
0         S. N.  Name                       
1         S. N.  Name                       
2           Badr Abdi              L.2169   
3          S. Acharya  Rajasthan \nL.2080   
4  Rajesh Kumar Verma                       

                                             Address Email  
0                                of the Life Members  None  
1                                of the Life Members  None  
2  L.2169 (96-97) 83/III Jai Jawan Colony JLN Mar...  None  
3  L.1120 (89-90) Shabd. Honorary Professor 33, S...  None  
4  L.2170 (91-92) 93-Nalapar, Purbia Tola Etawah, UP  None  


In [ ]:
df_cleaned = df

In [ ]:
# Drop rows where 'Name' is 'S. N. Name'
df = df[df['Name'] != 'S. N.  Name']

In [ ]:
print(df)

                          Name              Affiliation  \
2                    Badr Abdi                   L.2169   
3                   S. Acharya       Rajasthan \nL.2080   
4           Rajesh Kumar Verma                     None   
5    F.A. Saiyed Bagh E Ahmedi                     None   
6                 Ashok Parikh                    L.829   
..                         ...                      ...   
957       Yellukunte Extension        Bommanahalli Post   
958          Harikrishnan K.S.            IIT Kharagpur   
959              Apoorva Verma                     None   
960        Bhanu Pratap Pandey                     None   
961                Manu Sharma  Department of Economics   

                                               Address  \
2    L.2169 (96-97) 83/III Jai Jawan Colony JLN Mar...   
3    L.1120 (89-90) Shabd. Honorary Professor 33, S...   
4    L.2170 (91-92) 93-Nalapar, Purbia Tola Etawah, UP   
5    L.313 (74-75) 9, Roshan Park Society Jivaraj P...   
6

Save dataset

In [ ]:
# Save or display the updated DataFrame
df.to_csv("IES.csv", index=False)
print(df.head())


                        Name         Affiliation  \
2                  Badr Abdi              L.2169   
3                 S. Acharya  Rajasthan \nL.2080   
4         Rajesh Kumar Verma                None   
5  F.A. Saiyed Bagh E Ahmedi                None   
6               Ashok Parikh               L.829   

                                             Address Email  
2  L.2169 (96-97) 83/III Jai Jawan Colony JLN Mar...  None  
3  L.1120 (89-90) Shabd. Honorary Professor 33, S...  None  
4  L.2170 (91-92) 93-Nalapar, Purbia Tola Etawah, UP  None  
5  L.313 (74-75) 9, Roshan Park Society Jivaraj P...  None  
6  L.829 (85-86) Department of Economics Universi...  None  


# Scrapping ISI: 5th attempt: NLP 'en_core_web_sm'

## Process

In [ ]:
!pip install spacy pandas
import spacy
import pandas as pd

# Load the spaCy model
nlp = spacy.load("en_core_web_sm")

In [ ]:
import spacy
import pandas as pd
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Define a function to extract session details using NLP
def extract_session_details(text):
    doc = nlp(text)

    sessions = []

    # Regular expression to identify sessions and topics
    session_re = re.compile(r"Session (\d+[a-z]*)([^\n]*)\s*\(([^)]*)\)")

    # Regular expression to match the authors and affiliations
    author_re = re.compile(r"([A-Za-z\s]+)\s?\(([^)]+)\)")

    # Regular expression to capture paper title
    title_re = re.compile(r"\"([^\"]+)\"")

    # Iterate over all sessions in the text
    for match in session_re.finditer(text):
        session_number, session_topic, location = match.groups()

        # Locate the paper(s) associated with the session
        session_start = match.end()
        session_end = text.find("Session", session_start)  # Find next session to stop the current one
        session_content = text[session_start:session_end] if session_end != -1 else text[session_start:]

        # Extract the paper titles
        paper_titles = title_re.findall(session_content)

        # Extract authors and their affiliations
        authors_and_affiliations = author_re.findall(session_content)

        # Process each paper
        for paper_title in paper_titles:
            authors = []
            affiliations = []

            for author_name, affiliation in authors_and_affiliations:
                authors.append(author_name.strip())
                affiliations.append(affiliation.strip())

            # Loop over authors to create a row for each
            for author, affiliation in zip(authors, affiliations):
                sessions.append({
                    "Session": session_number,
                    "Topic": session_topic,
                    "Title of Paper": paper_title,
                    "Number of Authors": len(authors),
                    "Author": author,
                    "Institute/Affiliation": affiliation
                })

    # Convert the session details into a DataFrame
    df = pd.DataFrame(sessions)

    return df


# Extract session details from the conference text
conference_df2 = extract_session_details(text)

# Display the structured DataFrame
print(conference_df2)

# Save to CSV
conference_df2.to_csv("/content/drive/My Drive/conference_details_draft.csv", index=False)


     Session                   Topic  \
0          1  (a) Empirical Trade I    
1          1  (a) Empirical Trade I    
2          1  (a) Empirical Trade I    
3          1  (a) Empirical Trade I    
4          1  (a) Empirical Trade I    
...      ...                     ...   
1470       8        (i) Finance III    
1471       8        (i) Finance III    
1472       8        (i) Finance III    
1473       8        (i) Finance III    
1474       8        (i) Finance III    

                                         Title of Paper  Number of Authors  \
0     Credit Constraints, Bank Incentives, and Firm ...                  6   
1     Credit Constraints, Bank Incentives, and Firm ...                  6   
2     Credit Constraints, Bank Incentives, and Firm ...                  6   
3     Credit Constraints, Bank Incentives, and Firm ...                  6   
4     Credit Constraints, Bank Incentives, and Firm ...                  6   
...                                                

In [ ]:
import spacy
import pandas as pd
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Define a function to extract session details using NLP
def extract_session_details(text):
    doc = nlp(text)

    sessions = []

    # Regular expression to identify sessions and topics
    session_re = re.compile(r"Session (\d+[a-z]*)([^\n]*)\s*\(([^)]*)\)")

    # Regular expression to match the authors and affiliations
    author_re = re.compile(r"([A-Za-z\s]+)\s?\(([^)]+)\)")

    # Regular expression to capture paper title with 'begin paper' and 'end paper'
    paper_re = re.compile(r'begin paper(.*?)paper', re.DOTALL)

    # Iterate over all sessions in the text
    for match in session_re.finditer(text):
        session_number, session_topic, location = match.groups()

        # Locate the paper(s) associated with the session
        session_start = match.end()
        session_end = text.find("Session", session_start)  # Find next session to stop the current one
        session_content = text[session_start:session_end] if session_end != -1 else text[session_start:]

        # Extract the paper sections
        paper_sections = paper_re.findall(session_content)

        # Process each paper section
        for paper_section in paper_sections:
            authors = []
            affiliations = []

            # Extract the paper title
            title_match = re.search(r'\"([^\"]+)\"', paper_section)
            if title_match:
                paper_title = title_match.group(1)

            # Extract authors and their affiliations for the current paper
            for match in author_re.finditer(paper_section):
                author_name = match.group(1).strip()
                affiliation = match.group(2).strip()
                authors.append(author_name)
                affiliations.append(affiliation)

            # Loop over authors to create a row for each
            for author, affiliation in zip(authors, affiliations):
                sessions.append({
                    "Session": session_number,
                    "Topic": session_topic,
                    "Title of Paper": paper_title,
                    "Number of Authors": len(authors),
                    "Author": author,
                    "Institute/Affiliation": affiliation
                })

    # Convert the session details into a DataFrame
    df = pd.DataFrame(sessions)

    return df

# Apply the transformation to the input text
text_with_markers = add_paper_markers(text)

# Extract session details from the transformed conference text
conference_df = extract_session_details(text_with_markers)

# Display the structured DataFrame
print(conference_df)

# Save to CSV
conference_df.to_csv("/content/drive/My Drive/conference_details.csv", index=False)

    Session                   Topic  \
0         1  (a) Empirical Trade I    
1         1  (a) Empirical Trade I    
2         1  (a) Empirical Trade I    
3         1  (a) Empirical Trade I    
4         1  (a) Empirical Trade I    
..      ...                     ...   
450       8        (i) Finance III    
451       8        (i) Finance III    
452       8        (i) Finance III    
453       8        (i) Finance III    
454       8        (i) Finance III    

                                        Title of Paper  Number of Authors  \
0    Credit Constraints, Bank Incentives, and Firm ...                  2   
1    Credit Constraints, Bank Incentives, and Firm ...                  2   
2    Language Barriers in Multinationals and Knowle...                  4   
3    Language Barriers in Multinationals and Knowle...                  4   
4    Language Barriers in Multinationals and Knowle...                  4   
..                                                 ...               

Remove unwanted characters and separate authors

In [ ]:

import pandas as pd

# Load the conference details CSV
df = pd.read_csv("/content/drive/My Drive/conference_details.csv")

# Force 'Author' column to be in string format
df['Author'] = df['Author'].astype(str)

# Function to process the 'Author' column
def process_authors(df):
    expanded_rows = []
    for index, row in df.iterrows():
        authors = row['Author']

        # Remove 'and' at the beginning
        if authors.startswith('and'):
            authors = authors[3:].strip()

        # Split authors by 'and' and '\n' where appropriate
        split_authors = re.split(r'\band\b|\n', authors)
        split_authors = [author.strip() for author in split_authors if author.strip()]

        # Create a new row for each author
        for author in split_authors:
            new_row = row.copy()
            new_row['Author'] = author
            expanded_rows.append(new_row)

    # Create a new DataFrame with the expanded rows
    expanded_df = pd.DataFrame(expanded_rows)

    return expanded_df

# Process the authors and expand the DataFrame
expanded_df = process_authors(df)

# Display the expanded DataFrame
print(expanded_df)

# Save the expanded DataFrame to a new CSV
expanded_df.to_csv("/content/drive/My Drive/expanded_conference_details.csv", index=False)


     Session                   Topic  \
0          1  (a) Empirical Trade I    
1          1  (a) Empirical Trade I    
2          1  (a) Empirical Trade I    
3          1  (a) Empirical Trade I    
4          1  (a) Empirical Trade I    
..       ...                     ...   
450        8        (i) Finance III    
451        8        (i) Finance III    
452        8        (i) Finance III    
453        8        (i) Finance III    
454        8        (i) Finance III    

                                        Title of Paper  Number of Authors  \
0    Credit Constraints, Bank Incentives, and Firm ...                  2   
1    Credit Constraints, Bank Incentives, and Firm ...                  2   
2    Language Barriers in Multinationals and Knowle...                  4   
3    Language Barriers in Multinationals and Knowle...                  4   
4    Language Barriers in Multinationals and Knowle...                  4   
..                                                 ...   

Create unique authors and remove duplicates per Paper Title

In [ ]:
import pandas as pd

# Load the updated conference details CSV
df = pd.read_csv("/content/drive/My Drive/expanded_conference_details.csv")

# Function to clean the 'Title of Paper' column by removing new lines and merging with a space
def clean_titles(df):
    df['Title of Paper'] = df['Title of Paper'].str.replace('\n', ' ')
    return df

# Clean the 'Title of Paper' column
cleaned_df = clean_titles(df)

# Display the cleaned DataFrame to verify the changes
print(cleaned_df)


     Session                   Topic  \
0          1  (a) Empirical Trade I    
1          1  (a) Empirical Trade I    
2          1  (a) Empirical Trade I    
3          1  (a) Empirical Trade I    
4          1  (a) Empirical Trade I    
..       ...                     ...   
452        8        (i) Finance III    
453        8        (i) Finance III    
454        8        (i) Finance III    
455        8        (i) Finance III    
456        8        (i) Finance III    

                                        Title of Paper  Number of Authors  \
0    Credit Constraints, Bank Incentives, and Firm ...                  2   
1    Credit Constraints, Bank Incentives, and Firm ...                  2   
2    Language Barriers in Multinationals and Knowle...                  4   
3    Language Barriers in Multinationals and Knowle...                  4   
4    Language Barriers in Multinationals and Knowle...                  4   
..                                                 ...   

In [ ]:
print(cleaned_df)

     Session                   Topic  \
0          1  (a) Empirical Trade I    
1          1  (a) Empirical Trade I    
2          1  (a) Empirical Trade I    
3          1  (a) Empirical Trade I    
4          1  (a) Empirical Trade I    
..       ...                     ...   
452        8        (i) Finance III    
453        8        (i) Finance III    
454        8        (i) Finance III    
455        8        (i) Finance III    
456        8        (i) Finance III    

                                        Title of Paper  Number of Authors  \
0    Credit Constraints, Bank Incentives, and Firm ...                  2   
1    Credit Constraints, Bank Incentives, and Firm ...                  2   
2    Language Barriers in Multinationals and Knowle...                  4   
3    Language Barriers in Multinationals and Knowle...                  4   
4    Language Barriers in Multinationals and Knowle...                  4   
..                                                 ...   

## Save dataset

In [ ]:
# Save the cleaned and updated DataFrame to a new CSV
cleaned_df.to_csv("/content/drive/My Drive/ISI2022.csv", index=False)

# Experimental

In [ ]:
import spacy
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Function to process text with NLP and add markers
def add_markers_with_nlp(text):
    doc = nlp(text)
    updated_text = []
    delimiter = "***NEW PAPER***"

    session_re = re.compile(r'(Session \d+[a-z]* [^\n]* \([^\)]*\)\n)')
    session_blocks = session_re.split(text)

    for i, block in enumerate(session_blocks):
        if i % 2 == 0:
            updated_text.append(block)
        else:
            papers = re.split(r'(?:"[^"]+"\s*\n)', block)  # Split block at paper titles
            for j, paper in enumerate(papers):
                if j == 0:
                    updated_text.append(paper)
                else:
                    updated_text.append(f'{delimiter} "{papers[j-1].split(": ")[-1].strip()}" {paper}')

    return ''.join(updated_text)

# Assuming 'text' is already loaded
updated_text = add_markers_with_nlp(text)

# Print the updated text
print(updated_text)


Parallel Sessions 1                                                18 December, Monday 11:50 AM- 01:00  PM    
Session 1(a) Social Identity and Financial Inclusion (Conference Room) 
Rikhia Bhukta (Indian Institute of Technology Kanpur), Debayan Pakrashi (Indian Statistical Institute Kolkata; 
Indian Institute of Technology Kanpur), Sarani Saha (Indian Institute of Technology Kanpur), and Ashish Sedai 
(University of Texas at Arlington): "Does Financial Inclusion Mitigate Social Exclusion? Causal Evidence from 
India" 

Sandhya Garg (Institute of Economic Growth), Samarth Gupta (Ahmedabad University), and Sushanta Mallick 
(Queen Mary University of London): "Does Social Identity Constrain Rural Entrepreneurship? The Role of 
Financial Inclusion" 
Session 1(b) Welfare Programmes I (Seminar Room 1) 
Vidhya Unnikrishnan (University of Manchester): "Impact of Social Protection Policy Targeted on the 
Disabled: Panel Data Evidence from India" 


Michael Palmer (University of Western Austral

In [ ]:
import spacy
import pandas as pd
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Define a function to extract session details using NLP
def extract_session_details(text):
    doc = nlp(text)

    sessions = []

    # Regular expression to identify sessions and topics
    session_re = re.compile(r"Session (\d+[a-z]*)([^\n]*)\s*\(([^)]*)\)")

    # Regular expression to match the authors and affiliations
    author_re = re.compile(r"([A-Za-z\s]+)\s?\(([^)]+)\)")

    # Regular expression to capture paper titles
    title_re = re.compile(r'"\*\*\*NEW PAPER\*\*\* ([^\"]+)"')

    # Iterate over all sessions in the text
    for match in session_re.finditer(text):
        session_number, session_topic, location = match.groups()

        # Locate the paper(s) associated with the session
        session_start = match.end()
        session_end = text.find("Session", session_start)  # Find next session to stop the current one
        session_content = text[session_start:session_end] if session_end != -1 else text[session_start:]

        # Extract the paper titles
        paper_titles = title_re.findall(session_content)

        # Extract authors and their affiliations
        authors_and_affiliations = author_re.findall(session_content)

        # Process each paper
        for paper_title in paper_titles:
            authors = []
            affiliations = []

            for author_name, affiliation in authors_and_affiliations:
                authors.append(author_name.strip())
                affiliations.append(affiliation.strip())

            # Loop over authors to create a row for each
            for author, affiliation in zip(authors, affiliations):
                sessions.append({
                    "Session": session_number,
                    "Topic": session_topic,
                    "Title of Paper": paper_title,
                    "Number of Authors": len(authors),
                    "Author": author,
                    "Institute/Affiliation": affiliation
                })

    # Convert the session details into a DataFrame
    df = pd.DataFrame(sessions)

    return df

# Extract session details from the conference text
conference_df = extract_session_details(updated_text)

# Display the structured DataFrame
print(conference_df)

# Save to CSV
conference_df.to_csv("/content/drive/My Drive/conference_details.csv", index=False)


Empty DataFrame
Columns: []
Index: []
